# 01. First steps with OpenVINO — a face detection model on CPU · GPU · NPU

> Learn the flow that is the backbone of every lesson: **open the model → compile it for a device → preprocess → infer → postprocess**.
> Then learn how to measure speed properly by running the same model on each device.

### What you learn in this session
1. OpenVINO model files (IR: `.xml` + `.bin`) and `Core` · `read_model` · `compile_model`
2. Reading the model description: the names and shapes of the inputs · outputs
3. Preprocessing (photo → model input) and postprocessing (model output → boxes)
4. Measuring speed: compile time · first inference · average · throughput (async)

### The whole flow
```
photo (BGR, 480x640)  ──preprocess──▶  [1,3,300,300] float32  ──infer (CPU/GPU/NPU)──▶  [1,1,N,7]  ──postprocess──▶  face boxes
```

## 1. Setup — `Core` is the entrance to OpenVINO
One `ov.Core()` finds the devices on this PC and takes care of reading and compiling models.

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "../..")).resolve()       # program folder (lab.bat tells us)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # model folder
core = ov.Core()
print("Model folder:", MODELS)
print("Devices:", core.available_devices)

for d in core.available_devices:
    print(f"{d:6s}", core.get_property(d, "FULL_DEVICE_NAME"))

## 2. Opening the model — `read_model`
An OpenVINO model (IR) is two files.
- `.xml` — the structure: which calculations (operations) connect in which order
- `.bin` — the weights: the numbers decided by training

`read_model` reads the files and makes a model object that **does not depend on any device**. You cannot run it on a device yet.

The model we use: **face-detection-retail-0005** (Intel Open Model Zoo). It was trained for faces seen from the front at a medium distance, like from a store camera.

In [ ]:
MODEL = MODELS / "face" / "face-detection-retail-0005.xml"
assert MODEL.exists(), f"Model not found: {MODEL}"

model = core.read_model(MODEL)
inp, out = model.input(0), model.output(0)
print("Input:", inp.any_name, inp.partial_shape, inp.element_type)
print("Output:", out.any_name, out.partial_shape, out.element_type)
print("Number of operations:", len(model.get_ops()))

**Reading the model description**
- Input `[1, 3, 300, 300]` — **NCHW**: 1 photo · 3 color channels (**B, G, R** order) · height 300 · width 300. Values stay 0~255 as they are.
- Output `[1, 1, N, 7]` — N candidates (200 for this model), with 7 numbers for each candidate:

| Position | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| Meaning | image number | class (1=face) | score 0~1 | x1 | y1 | x2 | y2 |

  The coordinates are **ratios from 0 to 1** (0 = left/top edge, 1 = right/bottom edge). Multiply by the photo size to get pixels.
  If the image number is **-1**, the list ends there.
- `element_type` `f32` — you must feed float32.

This information is different for each model. In real work, first find and read the model description (model card). If you don't know, print it yourself like above.

## 3. One photo

In [ ]:
PHOTO = "sample.jpg"          # photo to use when there is no webcam (same folder as this notebook)

def show(bgr, width=720):
    """Show an OpenCV photo (BGR) inside the notebook."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """Grab one picture from the webcam. If there is no webcam, use the PHOTO file."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # some webcams give dark pictures for the first few frames
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"No webcam and no {PHOTO} either — put a photo file name in PHOTO"
        print("No webcam, so using the file:", PHOTO)
    return frame

frame = grab()
print("Photo size (height, width, channels):", frame.shape)
show(frame)

## 4. Preprocessing — turning the photo into the input shape
Of the five steps in section 6 of notebook 00, we use only ② size, ⑤ axis order and ④ number type. This model learned with BGR and takes 0~255, so we leave the color order and the number range as they are.

In [ ]:
def preprocess(bgr):
    x = cv2.resize(bgr, (300, 300))     # ① size: 300 x 300 (the shape gets squashed, but this model learned that way)
    x = x.transpose(2, 0, 1)            # ② order: (height, width, channels) → (channels, height, width)
    x = x[None].astype(np.float32)      # ③ add a 'how many photos' dimension at the front + float32
    return x

blob = preprocess(frame)
print(frame.shape, frame.dtype, "→", blob.shape, blob.dtype)

## 5. Compile · infer — `compile_model`
`compile_model(model, "CPU")` changes the model so it **runs as fast as possible on that device** (merging operations, arranging memory, making code for the device).
The result, a `CompiledModel`, can be called like a function: `compiled(input)` → a dictionary of outputs. You take an output out with a **port** such as `compiled.output(0)`.

In [ ]:
compiled = core.compile_model(model, "CPU")
res = compiled(blob)                         # one inference
dets = res[compiled.output(0)].reshape(-1, 7)  # flatten [1,1,N,7] → [N,7] so it is easy to work with
print("Number of candidates:", len(dets))
print("First 5 (number, class, score, x1, y1, x2, y2):")
print(np.round(dets[:5], 3))

## 6. Postprocessing — turning numbers into face boxes
1. If the image number is -1, stop (end of the list).
2. If the score is lower than the threshold (`conf`), throw it away.
3. Multiply the 0~1 coordinates by the photo's width · height to get pixels.

**What the score threshold means**: Lower it and you miss fewer faces, but odd things get picked up as faces too. Raise it and the opposite happens. Choose it based on what you need.

In [ ]:
def faces(dets, w, h, conf=0.5):
    boxes = []
    for image_id, label, score, x1, y1, x2, y2 in dets:
        if image_id < 0:                      # -1 = end of the candidate list
            break
        if score < conf:
            continue
        boxes.append((int(x1 * w), int(y1 * h), int(x2 * w), int(y2 * h), float(score)))
    return boxes

h, w = frame.shape[:2]
found = faces(dets, w, h)
view = frame.copy()                           # keep the original and draw on a copy
for x1, y1, x2, y2, s in found:
    cv2.rectangle(view, (x1, y1), (x2, y2), (0, 200, 0), 3)
    cv2.putText(view, f"{s:.2f}", (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 0), 2)
print("Faces found:", len(found), found)
show(view)

## 7. Measuring speed — the right way
The first run is slow (memory setup, device startup). So we measure three things separately.
- **Compile**: the time to turn the model into a version for that device. NPU · GPU can take a few seconds the first time → the cache in section 8 makes this shorter.
- **First inference**: one run right after compiling
- **Average**: the average of 100 runs after a few warm-up runs — use this number when you talk about real speed

`create_infer_request()` is a container that holds one inference job. Reusing it instead of making a new one each time is faster.

In [ ]:
def bench(device, n=100):
    t0 = time.perf_counter()
    c = core.compile_model(model, device)
    compile_s = time.perf_counter() - t0

    req = c.create_infer_request()
    t0 = time.perf_counter()
    req.infer({0: blob})
    first_ms = (time.perf_counter() - t0) * 1000

    for _ in range(5):                        # warm-up
        req.infer({0: blob})
    t0 = time.perf_counter()
    for _ in range(n):
        req.infer({0: blob})
    avg_ms = (time.perf_counter() - t0) / n * 1000
    return compile_s, first_ms, avg_ms

rows = []
for dev in core.available_devices:
    try:
        rows.append((dev, *bench(dev)))
    except Exception as ex:
        print(dev, "failed:", ex)

print(f"{'dev':6s}{'compile(s)':>10s}{'first(ms)':>12s}{'avg(ms)':>10s}{'FPS':>8s}")
for dev, cs, fm, am in rows:
    print(f"{dev:6s}{cs:10.2f}{fm:12.1f}{am:10.2f}{1000 / am:8.0f}")

In [ ]:
# Show as bars — shorter is faster
best = min(r[3] for r in rows)
for dev, cs, fm, am in rows:
    print(f"{dev:6s} {'█' * max(1, round(am / best * 10))} {am:.2f} ms")

**How to read it** — With a small model, the CPU may even come out fastest. The GPU · NPU have a cost for sending numbers to the device and back,
and when the calculation is small, that cost is bigger. On the other hand, the NPU uses **less power** and leaves the CPU free (good for doing other work at the same time).
"Which device is fastest" depends on the model size — let's compare again in notebook 02 (object detection with ed-m, a big model).

## 8. Compile cache — faster from the second time
If you set `CACHE_DIR`, the compile result is saved to disk, and next time it is just read back. This helps a lot on NPU · GPU.

In [ ]:
core.set_property({"CACHE_DIR": "ov-cache"})
dev = "NPU" if "NPU" in core.available_devices else ("GPU" if "GPU" in core.available_devices else "CPU")
for i in range(2):
    t0 = time.perf_counter()
    core.compile_model(model, dev)
    print(f"{dev} compile #{i + 1}: {(time.perf_counter() - t0) * 1000:.0f} ms")

## 9. One at a time vs many at once — async inference
So far we did **put in one photo → wait → next photo** (synchronous). When there are many photos, like in a video, putting several in **at the same time**
keeps the device busy, so the throughput per second (FPS) is higher. `AsyncInferQueue` does this job.
- **Latency**: the time until one photo comes out — how fast it reacts
- **Throughput**: how many photos per second — how much video it can handle

In [ ]:
c = core.compile_model(model, "CPU", {"PERFORMANCE_HINT": "THROUGHPUT"})   # compile for throughput
n_req = c.get_property("OPTIMAL_NUMBER_OF_INFER_REQUESTS")
queue = ov.AsyncInferQueue(c, n_req)
done = []
queue.set_callback(lambda request, userdata: done.append(userdata))       # called each time one photo is done

N = 200
t0 = time.perf_counter()
for i in range(N):
    queue.start_async({0: blob}, userdata=i)   # keep putting them in without waiting
queue.wait_all()
fps_async = N / (time.perf_counter() - t0)
print(f"{n_req} requests at once · {len(done)} photos · {fps_async:.0f} FPS (the sync average is the CPU FPS in the table above)")

## Common errors
| Symptom | Cause · fix |
|---|---|
| No faces found at all | Too far, a side view · too dark → try a lower `conf` or move closer. Skipping a preprocessing step (swapping the axes) also causes this |
| `Exception ... shape` | The input shape is wrong → check that `blob.shape` is `(1, 3, 300, 300)` |
| NPU compile fails | Driver · an unsupported operation → skip that device (`bench` prints "failed") |
| Boxes in the wrong place | You multiplied width and height the wrong way round → `w` for x, `h` for y |

## Summary
- The OpenVINO flow: `Core` → `read_model` (no device) → `compile_model` (for a device) → `compiled(input)` → take the result out with an output port.
- Preprocess exactly as the **model description** says: size · color order · number range · axis order.
- Measure speed as an average after warm-up. Latency and throughput are different. The cache cuts compile time.

## Things to try
1. Change it to `faces(dets, w, h, conf=0.3)` and to `conf=0.8`. What changes?
2. Compile with the device `"AUTO"` and see which device it picked.
   <details><summary>Hint</summary>

   `c = core.compile_model(model, "AUTO"); print(c.get_property("EXECUTION_DEVICES"))`
   </details>
3. Change section 9 to `"NPU"` · `"GPU"` and compare the throughput.
4. Count how many times per second it finds faces in the webcam video (keep `cv2.VideoCapture` open and repeat for 100 frames).
   <details><summary>Hint</summary>

   ```python
   cap = cv2.VideoCapture(0); t0 = time.perf_counter()
   for _ in range(100):
       ok, f = cap.read(); faces(compiled(preprocess(f))[compiled.output(0)].reshape(-1, 7), f.shape[1], f.shape[0])
   cap.release(); print(100 / (time.perf_counter() - t0), "FPS")
   ```
   </details>

---
# 🛠 Mini project — Face mosaic camera

**What you build**: A live camera that finds faces in the webcam video and **hides them with a mosaic** (privacy protection). It shows the FPS and the number of faces on the screen.
It runs for 15 seconds and saves the last frame to a file.

**What you use**: `preprocess` · `faces` (this notebook), the mosaic method from notebook 00, `live` (below — a function that shows video inside the notebook by swapping in each new frame)

In [ ]:
from IPython.display import Image as IPImage

def live(process, seconds=15, index=0, width=640):
    """Show webcam video in the notebook for `seconds` seconds, changing each frame with process(photo) → photo. Press I I to stop."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    still = None if cap.isOpened() and cap.read()[0] else cv2.imread(PHOTO)   # if there is no webcam, use the photo
    handle = display(None, display_id=True)
    t0, n, out = time.time(), 0, None
    try:
        while time.time() - t0 < seconds and (still is None or n < 10):
            if still is None:
                ok, f = cap.read()
                if not ok:
                    break
            else:
                f = still.copy()
            out = process(f)
            s = width / out.shape[1]
            small = cv2.resize(out, (width, int(out.shape[0] * s))) if s < 1 else out
            handle.update(IPImage(data=cv2.imencode(".jpg", small)[1].tobytes()))
            n += 1
    except KeyboardInterrupt:
        pass
    finally:
        cap.release()
    print(f"{n} frames · {n / max(time.time() - t0, 1e-6):.1f} FPS" + (" (no webcam, so faking it with the photo)" if still is not None else ""))
    return out

In [ ]:
best_dev = min(rows, key=lambda r: r[3])[0] if rows else "CPU"    # the fastest device from section 7
fast = core.compile_model(model, best_dev)
last = [time.perf_counter()]

def mosaic_faces(f):
    h, w = f.shape[:2]
    found = faces(fast(preprocess(f))[fast.output(0)].reshape(-1, 7), w, h, conf=0.5)
    for x1, y1, x2, y2, s in found:
        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w, x2), min(h, y2)
        roi = f[y1:y2, x1:x2]
        if roi.size:
            small = cv2.resize(roi, (max(1, roi.shape[1] // 12), max(1, roi.shape[0] // 12)))
            f[y1:y2, x1:x2] = cv2.resize(small, (roi.shape[1], roi.shape[0]), interpolation=cv2.INTER_NEAREST)
    now = time.perf_counter(); fps = 1 / max(now - last[0], 1e-6); last[0] = now
    cv2.putText(f, f"{best_dev} faces {len(found)}  {fps:.0f} FPS", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    return f

final = live(mosaic_faces, seconds=15)
if final is not None:
    cv2.imwrite("privacy_cam.jpg", final); print("Saved: privacy_cam.jpg")

**Try more**
- Instead of a mosaic: blur (`cv2.GaussianBlur`) · a black box · paste an emoji picture
- Hide faces **only when there is more than one person** (e.g. in a class recording, keep the one teacher visible)
- Change `best_dev` and compare the FPS